In [1]:
!apt-get update -qq
!apt-get install -y zstd
!which zstd
!zstd --version

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 95 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (3,076 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

In [4]:
!pip install ollama

In [5]:
from google.colab import files

uploaded = files.upload()
with open("img_1.md", "r", encoding="utf-8") as f:
    markdown = f.read()

Saving img_1.md to img_1.md


In [6]:
from google.colab import files



In [7]:
!ollama pull qwen2.5:7b

In [8]:
import ollama

prompt = f"""
You are converting a Brazilian electronic invoice (NF-e) into HTML.

Convert the following Markdown representation into a complete HTML document.

Requirements:
- Preserve all information from the input.
- Do not invent information.
- Preserve the hierarchy and organization of the invoice.
- Represent tables as HTML tables.
- Distinguish labels from their corresponding values when possible.
- Keep numeric values, dates, CNPJ, invoice numbers and product information exactly as provided.
- Return only the HTML code.
- Do not use Markdown code fences.

INPUT:

{markdown}
"""

response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

html = response["message"]["content"]

with open("img_1_rep.html", "w", encoding="utf-8") as f:
    f.write(html)

In [9]:
import json
from pathlib import Path

import ollama
from google.colab import files


uploaded = files.upload()
nome = next(iter(uploaded))
data = json.loads(uploaded[nome])

if isinstance(data, dict):
    data = data.get("res", data)

    blocos = [
        {
            "type": bloco.get("block_label"),
            "bbox": bloco.get("block_bbox"),
            "content": bloco.get("block_content"),
        }
        for bloco in data["parsing_res_list"]
    ]
else:
    blocos = data


prompt = f"""
Convert this OCR JSON from a Brazilian NF-e into a complete HTML document.

Requirements:
- preserve all information;
- keep labels associated with their values;
- preserve tables as HTML tables;
- do not change numbers, dates, CNPJ or product information;
- do not invent missing information;
- do not display bbox coordinates;
- use simple CSS;
- return only the HTML.

JSON:
{json.dumps(blocos, ensure_ascii=False)}
"""


response = ollama.chat(
    model="qwen2.5:7b",
    messages=[{"role": "user", "content": prompt}],
)

html = response["message"]["content"].strip()

html = html.removeprefix("```html")
html = html.removeprefix("```")
html = html.removesuffix("```").strip()

output = Path("nota_fiscal_json.html")
output.write_text(html, encoding="utf-8")

print(f"HTML salvo em {output}")

files.download(str(output))

Saving img_0_layout.json to img_0_layout.json
HTML salvo em nota_fiscal_json.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from IPython.display import display, HTML

with open("img_1_rep.html", "r", encoding="utf-8") as f:
    html_markdown = f.read()

display(HTML(html_markdown))

In [11]:
from IPython.display import IFrame

IFrame("nota_fiscal_json.html", width="100%", height=900)